# Customer Churn Prediction
### A Classification Approach to Identifying At-Risk Customers

---

**Project Goal:** Build a machine learning model that predicts whether a customer will leave (churn) based on their demographic profile, account details, and service usage — and turn those predictions into actionable retention strategies.

**Workflow at a Glance:**
1. Data Loading & Inspection
2. Data Cleaning & Preprocessing
3. Train-Test Split & Feature Scaling
4. Model Building & Evaluation
5. Predictions on New Unseen Data
6. Actionable Business Insights


---
## Step 0: Import Libraries

We import everything we need upfront so the rest of the notebook stays clean and focused.

In [ ]:
# --- Core data manipulation ---
import pandas as pd
import numpy as np

# --- Visualization ---
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# --- Scikit-learn: preprocessing ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- Scikit-learn: models ---
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# --- Scikit-learn: evaluation metrics ---
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# --- Suppress minor warnings to keep output readable ---
import warnings
warnings.filterwarnings('ignore')

# --- Make all plots look clean and consistent ---
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print('All libraries imported successfully!')

---
## Step 1: Data Loading & Basic Inspection

We load the telecom dataset directly from its public URL. Then we take a thorough first look:
- What does the data look like? (shape, columns, sample rows)
- Are there missing or oddly-typed values?
- How balanced is our target variable (Churn)?

In [ ]:
# The Google Sheets link is converted to a direct CSV export URL
url = 'https://docs.google.com/spreadsheets/d/1J2aMbLrRnk8g0Y5TSbz-en_7UxlI7xh0cLIZnIy4aew/export?format=csv'

df = pd.read_csv(url)

print(f'Dataset loaded successfully!')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n')
print('First 5 rows:')
df.head()

In [ ]:
# Check column names and data types
# This tells us which columns are numeric vs. text (object), and flags anything unexpected
print('Column names and data types:')
print(df.dtypes)

In [ ]:
# Check for missing values in each column
# A high number of nulls in an important column would require special attention
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values detected at this stage.')

In [ ]:
# Look at the distribution of our target variable: Churn
# This tells us if the dataset is balanced or skewed
churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

print('Target variable — Churn distribution:')
print(pd.DataFrame({'Count': churn_counts, 'Percentage (%)': churn_pct.round(1)}))

# Visualize the distribution
fig, ax = plt.subplots(figsize=(5, 3.5))
churn_counts.plot(kind='bar', ax=ax, color=['#4CAF50', '#F44336'], edgecolor='white', width=0.5)
ax.set_title('Churn Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Churn (No = 0, Yes = 1)', fontsize=10)
ax.set_ylabel('Number of Customers', fontsize=10)
ax.set_xticklabels(churn_counts.index, rotation=0)
for i, v in enumerate(churn_counts):
    ax.text(i, v + 30, f'{v:,}', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics for all numeric columns
# Mean, std, min/max help us spot outliers and understand the data range
print('Descriptive statistics for numeric columns:')
df.describe()

---
## Step 2: Data Cleaning & Preprocessing

Raw data is rarely ready to feed into a model. We:
- Fix `TotalCharges` (stored as text in the raw file — contains blank spaces)
- Drop `customerID` (a unique identifier that carries no predictive value)
- Encode the target variable `Churn` as 0/1
- One-hot encode all remaining categorical columns

In [ ]:
# TotalCharges contains blank strings ('  ') for some customers who just joined
# pd.to_numeric with errors='coerce' turns those blanks into NaN safely,
# then we fill NaN with the column's median — a robust central value
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

median_tc = df['TotalCharges'].median()
df['TotalCharges'].fillna(median_tc, inplace=True)

print(f'TotalCharges — NaN values after fix: {df["TotalCharges"].isnull().sum()}')
print(f'Filled missing values with median: {median_tc:.2f}')

In [ ]:
# customerID is just a unique identifier — it has no pattern the model can learn from,
# so keeping it would only add noise
if 'customerID' in df.columns:
    df.drop(columns=['customerID'], inplace=True)
    print('customerID column removed.')
else:
    print('customerID column not found — skipping.')

In [ ]:
# Encode the target: "Yes" → 1 (churned), "No" → 0 (retained)
# Machine learning models work with numbers, not text labels
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print('Churn values after encoding:')
print(df['Churn'].value_counts())

In [ ]:
# Identify categorical columns (object/text dtype) so we can one-hot encode them
# One-hot encoding converts each category into its own 0/1 column
# e.g., "Contract: Month-to-month" becomes a binary column that is 1 for month-to-month customers
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns to encode: {categorical_cols}\n')

# drop_first=True removes one redundant dummy per feature to avoid multicollinearity
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f'Dataset shape after encoding: {df.shape}')
print(f'Total features: {df.shape[1] - 1} (excluding target)')

In [ ]:
# Quick sanity check: confirm there are no remaining nulls in our clean dataset
print(f'Remaining nulls in dataset: {df.isnull().sum().sum()}')
print('\nFinal columns:')
print(df.columns.tolist())

---
## Step 3: Train-Test Split & Feature Scaling

We split the data into a **training set** (80%) to teach the model, and a **test set** (20%) to evaluate it on data it has never seen — this simulates real-world performance.

We then **standardize** numeric features (zero mean, unit variance) so that features with large magnitudes (like `TotalCharges`) don't dominate features with small magnitudes (like `SeniorCitizen`).

In [ ]:
# Separate features (X) from the target label (y)
X = df.drop(columns=['Churn'])
y = df['Churn']

# Split into training (80%) and testing (20%) sets
# random_state=42 ensures the same split every time we run the notebook
# stratify=y keeps the same Churn ratio in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]:,} rows')
print(f'Testing set:  {X_test.shape[0]:,} rows')
print(f'\nChurn ratio in training set: {y_train.mean():.2%}')
print(f'Churn ratio in testing set:  {y_test.mean():.2%}')

In [ ]:
# Identify numeric columns to scale
# We scale numeric features so large numbers don't overwhelm our model
# Example: MonthlyCharges (~70) vs. tenure (~30) vs. TotalCharges (~2000) — all on very different scales
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
# Only keep columns that actually exist in X after encoding
numeric_cols = [c for c in numeric_cols if c in X_train.columns]

scaler = StandardScaler()

# IMPORTANT: Fit the scaler ONLY on training data, then apply it to both train and test.
# If we fit on all data, the model would secretly "see" test set statistics — that's data leakage.
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])

print(f'Scaled columns: {numeric_cols}')
print('\nSample of scaled training data (first 3 rows):')
X_train[numeric_cols].head(3)

---
## Step 4: Model Building & Evaluation

We train two interpretable models:

| Model | Why it's useful |
|---|---|
| **Logistic Regression** | Fast, interpretable — gives us probability scores and clear feature weights |
| **Decision Tree** | Produces human-readable rules ("if tenure < 12 months AND contract is month-to-month, predict Churn") |

We evaluate each model using four key metrics:
- **Accuracy** — overall correctness
- **Precision** — of all customers we *predicted* to churn, how many actually did?
- **Recall** — of all customers who *actually* churned, how many did we catch?
- **F1-Score** — harmonic mean of Precision and Recall (great when classes are imbalanced)

In [ ]:
# --- Model 1: Logistic Regression ---
# max_iter=1000 ensures the solver converges on this dataset size
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# --- Model 2: Decision Tree ---
# max_depth=5 limits the tree so it doesn't memorize training data (overfitting)
# A shallow tree is also easier to interpret
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)

print('Both models trained successfully!')

In [ ]:
# Helper function so we don't repeat evaluation code for each model
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    cm   = confusion_matrix(y_test, y_pred)

    print(f'\n{'='*50}')
    print(f'  {name}')
    print(f'{'='*50}')
    print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    print(f'\n  Classification Report:')
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

    # Plot confusion matrix
    fig, ax = plt.subplots(figsize=(4.5, 3.5))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Predicted: No', 'Predicted: Yes'],
        yticklabels=['Actual: No', 'Actual: Yes'],
        ax=ax, linewidths=0.5, cbar=False
    )
    ax.set_title(f'Confusion Matrix — {name}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    return {'name': name, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

In [ ]:
# Evaluate Model 1: Logistic Regression
results_lr = evaluate_model('Logistic Regression', lr_model, X_test, y_test)

In [ ]:
# Evaluate Model 2: Decision Tree
results_dt = evaluate_model('Decision Tree (max_depth=5)', dt_model, X_test, y_test)

In [ ]:
# Side-by-side comparison of both models
metrics = ['accuracy', 'precision', 'recall', 'f1']
comparison = pd.DataFrame([results_lr, results_dt]).set_index('name')[metrics]
comparison.columns = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

print('\nModel Comparison:')
print(comparison.round(4).to_string())

# Visual comparison bar chart
ax = comparison.plot(kind='bar', figsize=(8, 4), width=0.6, edgecolor='white')
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.set_ylabel('Score', fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_xticklabels(comparison.index, rotation=10, ha='right')
ax.legend(loc='lower right', fontsize=9)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=7, padding=2)
plt.tight_layout()
plt.show()

In [ ]:
# Select the best model based on F1-Score — it's the most balanced metric
# when the dataset is slightly imbalanced (more No-churn than Churn customers)
if results_lr['f1'] >= results_dt['f1']:
    best_model = lr_model
    best_name  = 'Logistic Regression'
else:
    best_model = dt_model
    best_name  = 'Decision Tree'

print(f'Best model selected: {best_name}')
print(f'(F1-Score: LR={results_lr["f1"]:.4f} | DT={results_dt["f1"]:.4f})')

In [ ]:
# Feature importance: which factors drive churn the most?
# For Logistic Regression we use absolute coefficients as a proxy for importance
# For Decision Tree we use built-in feature_importances_

feature_names = X_train.columns.tolist()
n_show = 15  # Show top 15 features

if isinstance(best_model, LogisticRegression):
    importances = np.abs(best_model.coef_[0])
    importance_label = 'Absolute Coefficient (Logistic Regression)'
else:
    importances = best_model.feature_importances_
    importance_label = 'Feature Importance (Decision Tree)'

importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values('Importance', ascending=False).head(n_show)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=importance_df, y='Feature', x='Importance', ax=ax, palette='Blues_r')
ax.set_title(f'Top {n_show} Most Important Features\n({importance_label})', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance Score', fontsize=10)
ax.set_ylabel('')
plt.tight_layout()
plt.show()

print('\nTop 10 features driving churn prediction:')
print(importance_df.head(10).to_string(index=False))

---
## Step 5: Generating Predictions for New Unseen Data

In a real deployment, the model would score *new* incoming customer records — people not in the training or test set.

We simulate this by creating 3 fictional customers and running them through the **exact same preprocessing pipeline** as the training data. This step validates that our model is production-ready.

In [ ]:
# Create 3 sample new customers with realistic values
# These are fictional — not from the training or test set
new_customers_raw = pd.DataFrame({
    'gender':            ['Female', 'Male',   'Male'],
    'SeniorCitizen':     [0,         1,         0   ],
    'Partner':           ['Yes',    'No',      'Yes'],
    'Dependents':        ['No',     'No',      'Yes'],
    'tenure':            [2,         48,        24  ],    # months with the company
    'PhoneService':      ['Yes',    'Yes',     'No' ],
    'MultipleLines':     ['No',     'Yes',     'No phone service'],
    'InternetService':   ['Fiber optic', 'DSL', 'DSL'],
    'OnlineSecurity':    ['No',     'Yes',     'Yes'],
    'OnlineBackup':      ['No',     'No',      'Yes'],
    'DeviceProtection':  ['No',     'Yes',     'No' ],
    'TechSupport':       ['No',     'No',      'Yes'],
    'StreamingTV':       ['No',     'Yes',     'No' ],
    'StreamingMovies':   ['No',     'Yes',     'No' ],
    'Contract':          ['Month-to-month', 'Two year', 'One year'],
    'PaperlessBilling':  ['Yes',    'No',      'Yes'],
    'PaymentMethod':     ['Electronic check', 'Bank transfer (automatic)', 'Mailed check'],
    'MonthlyCharges':    [79.85,    55.90,     43.20],
    'TotalCharges':      [159.70,   2683.20,   1036.80],
})

print('New customer profiles:')
new_customers_raw

In [ ]:
# ----- Preprocess new customers using the SAME steps as the training data -----

new_customers = new_customers_raw.copy()

# Step A: Fix TotalCharges (apply same conversion, just in case)
new_customers['TotalCharges'] = pd.to_numeric(new_customers['TotalCharges'], errors='coerce')
new_customers['TotalCharges'].fillna(median_tc, inplace=True)

# Step B: One-hot encode categorical columns
new_customers_encoded = pd.get_dummies(new_customers, drop_first=True)

# Step C: Align columns with training data
# New data may be missing some dummy columns that existed in training
# We add them as 0 (absent category) to maintain the exact same feature space
new_customers_aligned = new_customers_encoded.reindex(columns=X_train.columns, fill_value=0)

# Step D: Apply the SAME scaler fitted on training data (do NOT refit)
new_customers_aligned[numeric_cols] = scaler.transform(new_customers_aligned[numeric_cols])

print(f'New customers preprocessed. Shape: {new_customers_aligned.shape}')
print('Feature count matches training data:', new_customers_aligned.shape[1] == X_train.shape[1])

In [ ]:
# Generate predictions and probability scores for each new customer
predictions  = best_model.predict(new_customers_aligned)
probabilities = best_model.predict_proba(new_customers_aligned)[:, 1]  # probability of churning

results_new = pd.DataFrame({
    'Customer':         ['Customer A', 'Customer B', 'Customer C'],
    'Tenure (months)':  new_customers_raw['tenure'].values,
    'Contract Type':    new_customers_raw['Contract'].values,
    'Monthly Charges':  new_customers_raw['MonthlyCharges'].values,
    'Predicted Churn':  ['Yes 🔴' if p == 1 else 'No 🟢' for p in predictions],
    'Churn Probability': [f'{p:.1%}' for p in probabilities],
    'Risk Level':       ['High Risk' if p > 0.65 else 'Medium Risk' if p > 0.35 else 'Low Risk'
                         for p in probabilities]
})

print(f'Predictions using: {best_name}\n')
print(results_new.to_string(index=False))

In [ ]:
# Visualize churn probability for each new customer
customers   = results_new['Customer'].tolist()
probs       = [float(p.strip('%')) / 100 for p in results_new['Churn Probability'].tolist()]
colors      = ['#F44336' if p > 0.5 else '#4CAF50' for p in probs]

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.barh(customers, probs, color=colors, edgecolor='white', height=0.5)
ax.axvline(x=0.5, color='gray', linestyle='--', linewidth=1.2, label='Decision threshold (0.5)')
ax.set_xlim(0, 1)
ax.set_xlabel('Probability of Churning', fontsize=10)
ax.set_title('Churn Probability for New Customers', fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
for bar, prob in zip(bars, probs):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{prob:.1%}', va='center', fontsize=10, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## Step 6: Actionable Business Insights

A model is only valuable if it drives real-world decisions. Based on the top features identified by our model, here are three concrete recommendations for the business.

In [ ]:
# Reload the original clean data (before encoding) for business-level charts
# We load fresh so column names are still human-readable
df_raw = pd.read_csv(url)
df_raw['TotalCharges'] = pd.to_numeric(df_raw['TotalCharges'], errors='coerce')
df_raw['TotalCharges'].fillna(df_raw['TotalCharges'].median(), inplace=True)
df_raw['Churn_Label'] = df_raw['Churn']  # keep original 'Yes'/'No' for chart labels

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# --- Chart 1: Churn rate by Contract Type ---
contract_churn = df_raw.groupby('Contract')['Churn_Label'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
contract_churn.columns = ['Contract', 'Churn Rate (%)']
contract_churn = contract_churn.sort_values('Churn Rate (%)', ascending=False)
sns.barplot(data=contract_churn, x='Contract', y='Churn Rate (%)', palette='Reds_r', ax=axes[0])
axes[0].set_title('Insight 1: Contract Type vs Churn Rate', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Contract Type', fontsize=9)
axes[0].set_ylabel('Churn Rate (%)', fontsize=9)
axes[0].set_xticklabels(contract_churn['Contract'], rotation=10, ha='right', fontsize=8)
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=9, fontweight='bold')

# --- Chart 2: Churn by Tenure buckets ---
df_raw['Tenure Bucket'] = pd.cut(df_raw['tenure'], bins=[0, 12, 24, 36, 48, 72],
                                  labels=['0–12 mo', '13–24 mo', '25–36 mo', '37–48 mo', '49–72 mo'])
tenure_churn = df_raw.groupby('Tenure Bucket', observed=True)['Churn_Label'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
tenure_churn.columns = ['Tenure Bucket', 'Churn Rate (%)']
sns.barplot(data=tenure_churn, x='Tenure Bucket', y='Churn Rate (%)', palette='Blues_r', ax=axes[1])
axes[1].set_title('Insight 2: Tenure vs Churn Rate', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Customer Tenure', fontsize=9)
axes[1].set_ylabel('Churn Rate (%)', fontsize=9)
axes[1].set_xticklabels(tenure_churn['Tenure Bucket'], rotation=10, ha='right', fontsize=8)
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=9, fontweight='bold')

# --- Chart 3: Monthly Charges distribution by Churn ---
sns.boxplot(data=df_raw, x='Churn_Label', y='MonthlyCharges', palette={'No': '#4CAF50', 'Yes': '#F44336'},
            ax=axes[2], width=0.5)
axes[2].set_title('Insight 3: Monthly Charges vs Churn', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Churned?', fontsize=9)
axes[2].set_ylabel('Monthly Charges ($)', fontsize=9)

plt.suptitle('Key Business Insights from Model Features', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 💡 Recommendation 1 — Incentivize Long-Term Contracts

**Insight:** Month-to-month contract customers have a churn rate significantly higher than one-year or two-year contract customers. The flexibility of short-term contracts reduces switching costs for dissatisfied customers.

**Action:** Offer a discount or loyalty bonus (e.g., one free month, upgraded speed, free add-on) to month-to-month customers who commit to a 12-month or 24-month plan. Target this offer to customers with `tenure < 12 months` and `MonthlyCharges > $60`, as they are in the highest-risk segment.

---

### 💡 Recommendation 2 — Invest Heavily in the Critical First 12 Months

**Insight:** Churn is highest in the first year of a customer's relationship with the company. New customers haven't yet experienced enough value to build loyalty.

**Action:** Launch a structured "First-Year Success Program": onboarding calls at months 1 and 3, proactive check-in emails at month 6, and an anniversary reward at month 12. Track Net Promoter Score (NPS) at each touchpoint and trigger a retention specialist intervention when NPS drops below a threshold.

---

### 💡 Recommendation 3 — Review Pricing for High-Paying, At-Risk Customers

**Insight:** Customers who churn tend to pay higher monthly charges than those who stay. This suggests a perceived value gap — they're paying more but not feeling it's worth it.

**Action:** For customers with `MonthlyCharges > $75` and `tenure < 24 months`, conduct a quarterly pricing review. Consider bundling additional services (e.g., TechSupport + StreamingTV) at no extra cost rather than discounting. Adding value is stickier than reducing price, and it avoids training customers to wait for discounts.


---
## Step 7: Export Predictions to CSV

We save the new customer predictions to `outputs/new_customer_predictions.csv` so the results can be shared with business teams, loaded into dashboards, or archived as a project deliverable.

This mirrors how a real ML pipeline would hand off scored records to a CRM or retention tool.

In [ ]:
import os

# Define 5 realistic new customer profiles covering a range of risk scenarios
new_customers_export = pd.DataFrame({
    'Customer_ID':       ['CUST_001',         'CUST_002',                     'CUST_003',     'CUST_004',          'CUST_005'],
    'Gender':            ['Female',            'Male',                         'Male',          'Female',            'Male'],
    'SeniorCitizen':     [0,                   1,                              0,               0,                   0],
    'Partner':           ['Yes',               'No',                           'Yes',           'No',                'Yes'],
    'Dependents':        ['No',                'No',                           'Yes',           'No',                'No'],
    'Tenure_Months':     [2,                   48,                             24,              5,                   14],
    'PhoneService':      ['Yes',               'Yes',                          'No',            'Yes',               'Yes'],
    'MultipleLines':     ['No',                'Yes',                          'No phone service', 'Yes',            'No'],
    'InternetService':   ['Fiber optic',       'DSL',                          'DSL',           'Fiber optic',       'Fiber optic'],
    'OnlineSecurity':    ['No',                'Yes',                          'Yes',           'No',                'No'],
    'OnlineBackup':      ['No',                'No',                           'Yes',           'No',                'Yes'],
    'DeviceProtection':  ['No',                'Yes',                          'No',            'No',                'No'],
    'TechSupport':       ['No',                'No',                           'Yes',           'No',                'No'],
    'StreamingTV':       ['No',                'Yes',                          'No',            'Yes',               'No'],
    'StreamingMovies':   ['No',                'Yes',                          'No',            'Yes',               'No'],
    'Contract':          ['Month-to-month',    'Two year',                     'One year',      'Month-to-month',    'Month-to-month'],
    'PaperlessBilling':  ['Yes',               'No',                           'Yes',           'Yes',               'Yes'],
    'PaymentMethod':     ['Electronic check',  'Bank transfer (automatic)',     'Mailed check',  'Electronic check',  'Credit card (automatic)'],
    'MonthlyCharges':    [79.85,               55.90,                          43.20,           89.10,               61.40],
    'TotalCharges':      [159.70,              2683.20,                        1036.80,         445.50,              859.60],
})

# ── Preprocessing: apply the exact same pipeline as the training data ─────────

new_exp = new_customers_export.drop(columns=['Customer_ID']).copy()

# Fix TotalCharges (same coerce-and-fill approach as training)
new_exp['TotalCharges'] = pd.to_numeric(new_exp['TotalCharges'], errors='coerce')
new_exp['TotalCharges'].fillna(median_tc, inplace=True)

# One-hot encode — drop_first=True matches training encoding
new_exp_encoded = pd.get_dummies(new_exp, drop_first=True)

# Align to the exact feature columns from training (fill missing dummies with 0)
new_exp_aligned = new_exp_encoded.reindex(columns=X_train.columns, fill_value=0)

# Scale numeric features using the SAME fitted scaler — never refit on new data
new_exp_aligned[numeric_cols] = scaler.transform(new_exp_aligned[numeric_cols])

# ── Generate predictions ──────────────────────────────────────────────────────

export_preds  = best_model.predict(new_exp_aligned)            # 0 or 1
export_probs  = best_model.predict_proba(new_exp_aligned)[:, 1]  # probability of churn
export_risk   = ['High' if p > 0.65 else 'Medium' if p > 0.35 else 'Low' for p in export_probs]

# ── Build the output DataFrame ────────────────────────────────────────────────

output_df = new_customers_export[['Customer_ID', 'Tenure_Months', 'Contract', 'MonthlyCharges', 'TotalCharges']].copy()
output_df['Predicted_Churn']    = export_preds
output_df['Churn_Probability']  = export_probs.round(4)
output_df['Risk_Level']         = export_risk

print('Predictions for new customers:')
print(output_df.to_string(index=False))

# ── Save to CSV ───────────────────────────────────────────────────────────────

# Create the outputs/ directory if it doesn't exist yet
os.makedirs('../outputs', exist_ok=True)

output_path = '../outputs/new_customer_predictions.csv'
output_df.to_csv(output_path, index=False)

print(f'\nPredictions saved to: {output_path}')

---
## Summary

| Step | What we did |
|---|---|
| Data Loading | Loaded 7,000+ customer records from a public Google Sheets URL |
| Preprocessing | Fixed `TotalCharges`, removed ID column, encoded target & categoricals |
| Splitting & Scaling | 80/20 stratified split, StandardScaler on numeric features |
| Modeling | Trained Logistic Regression and Decision Tree; selected best by F1-Score |
| New Predictions | Scored 3 new fictional customers with churn probability and risk level |
| Business Insights | Derived 3 actionable retention strategies from the model's top features |

**Key takeaway:** Tenure, contract type, and monthly charges are the strongest predictors of churn. Customers in their first year, on month-to-month contracts, paying above-average fees represent the highest-priority retention targets.

---
*This notebook is production-ready for a GitHub portfolio. All code is self-contained and reproducible.*